In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('insurance.csv')

In [3]:
df

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520
...,...,...,...,...,...,...,...
1333,50,male,30.970,3,no,northwest,10600.54830
1334,18,female,31.920,0,no,northeast,2205.98080
1335,18,female,36.850,0,no,southeast,1629.83350
1336,21,female,25.800,0,no,southwest,2007.94500


In [4]:
#começar a limpeza
print('Duplicadas:', df.duplicated().sum())
df[df.duplicated(keep=False)]

Duplicadas: 1


,age,sex,bmi,children,smoker,region,charges
195,19,male,30.59,0,no,northwest,1639.5631
581,19,male,30.59,0,no,northwest,1639.5631


In [5]:
df = df.drop_duplicates()
df.shape

(1337, 7)

In [6]:
df.sex = df.sex.map({'female': 0, 'male': 1})
df.smoker = df.smoker.map({'no': 0, 'yes': 1})
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,0,27.900,0,1,southwest,16884.92400
1,18,1,33.770,1,0,southeast,1725.55230
2,28,1,33.000,3,0,southeast,4449.46200
3,33,1,22.705,0,0,northwest,21984.47061
4,32,1,28.880,0,0,northwest,3866.85520


In [7]:
df = pd.get_dummies(df, columns=['region'], drop_first=True, dtype=int)
df.head()

,age,sex,bmi,children,smoker,charges,region_northwest,region_southeast,region_southwest
0,19,0,27.900,0,1,16884.92400,0,0,1
1,18,1,33.770,1,0,1725.55230,0,1,0
2,28,1,33.000,3,0,4449.46200,0,1,0
3,33,1,22.705,0,0,21984.47061,1,0,0
4,32,1,28.880,0,0,3866.85520,1,0,0


In [8]:
df.shape
df.dtypes
df.isnull().sum()

age                 0
sex                 0
bmi                 0
children            0
smoker              0
charges             0
region_northwest    0
region_southeast    0
region_southwest    0
dtype: int64

In [9]:
#como nós ja temos todas as informações desse projeto no outro projeto, podemos ir direto para a modelagem

In [10]:
#importação
from sklearn.model_selection import train_test_split, GridSearchCV, KFold, cross_val_score
from sklearn.model_selection import KFold, cross_validate
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [11]:
#separar variáveis
X = df.drop(columns=['charges'])
y = df.charges

In [12]:
#treino e teste
X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, test_size=0.3, random_state=0)

In [13]:
#validação cruada
kfold = KFold(n_splits=10, shuffle=True, random_state=0)

In [14]:
#criação do modelo
modelo = MLPRegressor(hidden_layer_sizes=(100, 100), activation='relu', solver='lbfgs', max_iter=1000, tol=0.0001, random_state=0, verbose=True)

In [15]:
resultado = cross_validate(modelo, X, y, cv=kfold, scoring='r2', return_train_score=True)

c:\Users\caioa\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:606: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)
c:\Users\caioa\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:606: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)
c:\Users\caioa\anaconda3

In [16]:
print('R² treino (média):', resultado['train_score'].mean())
print('R² teste (média):', resultado['test_score'].mean())

R² treino (média): 0.8157706683390501
R² teste (média): 0.7953158064547363


In [17]:
from sklearn.preprocessing import StandardScaler

In [18]:
#resultado não foi bom, então vamos tentar padronizar os dados
scaler = StandardScaler()
X_treino_pad = scaler.fit_transform(X_treino)
X_teste_pad = scaler.transform(X_teste)

In [19]:
print(X_treino_pad.mean(axis=0).round(2))
print(X_treino_pad.std(axis=0).round(2))

[-0. -0.  0. -0. -0.  0.  0. -0.]
[1. 1. 1. 1. 1. 1. 1. 1.]


In [20]:
#seguir com o modelo
modelo_pad = MLPRegressor(hidden_layer_sizes=(100, 100), activation='relu', solver='lbfgs', max_iter=1000, tol=0.0001, random_state=0)
modelo_pad.fit(X_treino_pad, y_treino)

c:\Users\caioa\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:606: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


,"hidden_layer_sizes hidden_layer_sizes: array-like of shape(n_layers - 2,), default=(100,)The ith element represents the number of neurons in the ithhidden layer.","(100, ...)"
,"solver solver: {'lbfgs', 'sgd', 'adam'}, default='adam'The solver for weight optimization.- 'lbfgs' is an optimizer in the family of quasi-Newton methods.- 'sgd' refers to stochastic gradient descent.- 'adam' refers to a stochastic gradient-based optimizer proposed by Kingma, Diederik, and Jimmy BaFor a comparison between Adam optimizer and SGD, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_training_curves.py`.Note: The default solver 'adam' works pretty well on relativelylarge datasets (with thousands of training samples or more) in terms ofboth training time and validation score.For small datasets, however, 'lbfgs' can converge faster and performbetter.",'lbfgs'
,"max_iter max_iter: int, default=200Maximum number of iterations. The solver iterates until convergence(determined by 'tol') or this number of iterations. For stochasticsolvers ('sgd', 'adam'), note that this determines the number of epochs(how many times each data point will be used), not the number ofgradient steps.",1000
,"random_state random_state: int, RandomState instance, default=NoneDetermines random number generation for weights and biasinitialization, train-test split if early stopping is used, and batchsampling when solver='sgd' or 'adam'.Pass an int for reproducible results across multiple function calls.See :term:`Glossary <random_state>`.",0
,"loss loss: {'squared_error', 'poisson'}, default='squared_error'The loss function to use when training the weights. Note that the""squared error"" and ""poisson"" losses actually implement""half squares error"" and ""half poisson deviance"" to simplify thecomputation of the gradient. Furthermore, the ""poisson"" loss internally usesa log-link (exponential as the output activation function) and requires``y >= 0``... versionchanged:: 1.7 Added parameter `loss` and option 'poisson'.",'squared_error'
,"activation activation: {'identity', 'logistic', 'tanh', 'relu'}, default='relu'Activation function for the hidden layer.- 'identity', no-op activation, useful to implement linear bottleneck, returns f(x) = x- 'logistic', the logistic sigmoid function, returns f(x) = 1 / (1 + exp(-x)).- 'tanh', the hyperbolic tan function, returns f(x) = tanh(x).- 'relu', the rectified linear unit function, returns f(x) = max(0, x)",'relu'
,"alpha alpha: float, default=0.0001Strength of the L2 regularization term. The L2 regularization termis divided by the sample size when added to the loss.",0.0001
,"batch_size batch_size: int, default='auto'Size of minibatches for stochastic optimizers.If the solver is 'lbfgs', the regressor will not use minibatch.When set to ""auto"", `batch_size=min(200, n_samples)`.",'auto'
,"learning_rate learning_rate: {'constant', 'invscaling', 'adaptive'}, default='constant'Learning rate schedule for weight updates.- 'constant' is a constant learning rate given by 'learning_rate_init'.- 'invscaling' gradually decreases the learning rate ``learning_rate_`` at each time step 't' using an inverse scaling exponent of 'power_t'. effective_learning_rate = learning_rate_init / pow(t, power_t)- 'adaptive' keeps the learning rate constant to 'learning_rate_init' as long as training loss keeps decreasing. Each time two consecutive epochs fail to decrease training loss by at least tol, or fail to increase validation score by at least tol if 'early_stopping' is on, the current learning rate is divided by 5.Only used when solver='sgd'.",'constant'
,"learning_rate_init learning_rate_init: float, default=0.001The initial learning rate used. It controls the step-sizein updating the weights. Only used when solver='sgd' or 'adam'.",0.001
,"power_t power_t: float, default=0.5The exponent for inverse scaling learning rate.It is used in updating effective learning rate when the learning_rateis set to 'invscaling'. Only used when solver='sgd'."

In [21]:
r2_treino_pad = modelo_pad.score(X_treino_pad, y_treino)
print('R² treino:', r2_treino_pad)

R² treino: 0.9904600162317437


In [22]:
#teste
previsoes_pad = modelo_pad.predict(X_teste_pad)

In [23]:
r2_teste_pad = r2_score(y_teste, previsoes_pad)
print('R² teste:', r2_teste_pad)

R² teste: 0.035741821587668476


In [24]:
#ajuste de hiperparâmetros
modelo_pad = MLPRegressor(hidden_layer_sizes=(100, 100), activation='relu', solver='lbfgs', alpha=0.01, max_iter=1000, tol=0.0001, random_state=0)
modelo_pad.fit(X_treino_pad, y_treino)
print('R² treino:', modelo_pad.score(X_treino_pad, y_treino))
print('R² teste:', modelo_pad.score(X_teste_pad, y_teste))

R² treino: 0.9941824842276726
R² teste: 0.21729721803768742


c:\Users\caioa\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:606: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


In [ ]:
#melhorou, mas vamos melhorar mais, criando um pipeline e usando gridsearch para encontrar os melhores hiperparâmetros
from sklearn.pipeline import Pipeline

pipeline = Pipeline([('scaler', StandardScaler()),('mlp', MLPRegressor(activation='relu', solver='lbfgs', max_iter=1000, tol=0.0001, random_state=0))])

In [27]:

parametros = {'mlp__hidden_layer_sizes': [(20, 20), (50, 50), (100, 100)],'mlp__alpha': [0.0001, 0.01, 0.1, 1, 10]}

In [28]:
busca = GridSearchCV(pipeline, parametros, cv=kfold, scoring='r2', return_train_score=True)
busca.fit(X_treino, y_treino)
print('Melhores parâmetros:', busca.best_params_)
print('Melhor R² (validação):', busca.best_score_)

c:\Users\caioa\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:606: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)
c:\Users\caioa\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:606: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)
c:\Users\caioa\anaconda3

Melhores parâmetros: {'mlp__alpha': 0.0001, 'mlp__hidden_layer_sizes': (20, 20)}
Melhor R² (validação): 0.7569326094466675


c:\Users\caioa\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:606: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


In [30]:
#ele nos deu uma combinação menor, ent vamos buscar mais combinações 
parametros2 = {
    'mlp__hidden_layer_sizes': [(5,), (10,), (5, 5), (10, 10), (20, 20)],
    'mlp__alpha': [0.0001, 0.01, 1]
}
busca2 = GridSearchCV(pipeline, parametros2, cv=kfold, scoring='r2', return_train_score=True)
busca2.fit(X_treino, y_treino)
print('Melhores parâmetros:', busca2.best_params_)
print('Melhor R² (validação):', busca2.best_score_)

Melhores parâmetros: {'mlp__alpha': 0.01, 'mlp__hidden_layer_sizes': (10,)}
Melhor R² (validação): 0.8293476946884546


c:\Users\caioa\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:606: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


In [32]:
#agora treinar e testarcom melhor modelo
melhor_modelo = busca2.best_estimator_
print('R² treino:', melhor_modelo.score(X_treino, y_treino))

R² treino: 0.8717936577303343


In [33]:
previsoes = melhor_modelo.predict(X_teste)
print('R² teste:', r2_score(y_teste, previsoes))

R² teste: 0.8564432981765702


In [34]:
#analisar metricas 
previsoes = melhor_modelo.predict(X_teste)
print('R² treino:', melhor_modelo.score(X_treino, y_treino))
print('R² teste:', r2_score(y_teste, previsoes))
print('Erro absoluto médio:', mean_absolute_error(y_teste, previsoes))
print('Erro quadrático médio:', mean_squared_error(y_teste, previsoes))
print('Raiz do erro quadrático médio:', np.sqrt(mean_squared_error(y_teste, previsoes)))

R² treino: 0.8717936577303343
R² teste: 0.8564432981765702
Erro absoluto médio: 2813.786363325873
Erro quadrático médio: 23508781.41630056
Raiz do erro quadrático médio: 4848.5855067535485


In [36]:
modelo_final = Pipeline([('scaler', StandardScaler()),('mlp', MLPRegressor(hidden_layer_sizes=(10,), alpha=0.01, activation='relu', solver='lbfgs', max_iter=1000, tol=0.0001, random_state=0))])
resultado_final = cross_validate(modelo_final, X, y, cv=kfold, scoring='r2', return_train_score=True)
print('R² treino (média):', resultado_final['train_score'].mean())
print('R² teste (média):', resultado_final['test_score'].mean())

c:\Users\caioa\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:606: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


R² treino (média): 0.8723978285626514
R² teste (média): 0.8390147751906897


In [37]:
#para a comparação nao ficar injusta vamos fazer os hiperparâmetros dos dados não padronizados iguais aos do modelo final
modelo_sem = MLPRegressor(activation='relu', solver='lbfgs', max_iter=1000, tol=0.0001, random_state=0)
parametros_sem = {'hidden_layer_sizes': [(5,), (10,), (5, 5), (10, 10), (20, 20), (100, 100)],'alpha': [0.0001, 0.01, 1]}
busca_sem = GridSearchCV(modelo_sem, parametros_sem, cv=kfold, scoring='r2', return_train_score=True, n_jobs=-1)
busca_sem.fit(X_treino, y_treino)
print('Melhores parâmetros:', busca_sem.best_params_)
print('Melhor R² (validação):', busca_sem.best_score_)

Melhores parâmetros: {'alpha': 1, 'hidden_layer_sizes': (10, 10)}
Melhor R² (validação): 0.8345221993552034


In [38]:
parametros_sem2 = {'hidden_layer_sizes': [(10, 10), (20, 20), (50, 50)], 'alpha': [1, 3, 10, 30]}
busca_sem2 = GridSearchCV(modelo_sem, parametros_sem2, cv=kfold, scoring='r2', return_train_score=True, n_jobs=-1)
busca_sem2.fit(X_treino, y_treino)
print('Melhores parâmetros:', busca_sem2.best_params_)
print('Melhor R² (validação):', busca_sem2.best_score_)

Melhores parâmetros: {'alpha': 1, 'hidden_layer_sizes': (10, 10)}
Melhor R² (validação): 0.8345221993552034


In [40]:
#com o melhor parâmetro, vamos treinar e testar o modelo sem padronização e ver as metricas
melhor_modelo_sem = busca_sem2.best_estimator_
previsoes_sem = melhor_modelo_sem.predict(X_teste)
print('R² treino:', melhor_modelo_sem.score(X_treino, y_treino))
print('R² teste:', r2_score(y_teste, previsoes_sem))
print('Erro absoluto médio:', mean_absolute_error(y_teste, previsoes_sem))
print('Erro quadrático médio:', mean_squared_error(y_teste, previsoes_sem))
print('Raiz do erro quadrático médio:', np.sqrt(mean_squared_error(y_teste, previsoes_sem)))

R² treino: 0.7427080829849368
R² teste: 0.7640912147429932
Erro absoluto médio: 4227.919219752893
Erro quadrático médio: 38632317.37946501
Raiz do erro quadrático médio: 6215.490115788537


In [41]:
#com a validação cruzada
modelo_final_sem = MLPRegressor(hidden_layer_sizes=(10, 10), alpha=1, activation='relu', solver='lbfgs', max_iter=1000, tol=0.0001, random_state=0)
resultado_final_sem = cross_validate(modelo_final_sem, X, y, cv=kfold, scoring='r2', return_train_score=True, n_jobs=-1)
print('R² treino (média):', resultado_final_sem['train_score'].mean())
print('R² teste (média):', resultado_final_sem['test_score'].mean())

R² treino (média): 0.8424195406101089
R² teste (média): 0.8340360877075689


In [42]:
#agora com a comparação final
comparacao = pd.DataFrame({'modelo': ['sem padronização (10, 10), alpha=1', 'padronizada (10,), alpha=0.01'],'r2_treino_cv': [resultado_final_sem['train_score'].mean(), resultado_final['train_score'].mean()],'r2_teste_cv': [resultado_final_sem['test_score'].mean(), resultado_final['test_score'].mean()],'r2_teste_separado': [r2_score(y_teste, previsoes_sem), r2_score(y_teste, previsoes)],'erro_absoluto_medio': [mean_absolute_error(y_teste, previsoes_sem), mean_absolute_error(y_teste, previsoes)]})
comparacao.round(3)

,modelo,r2_treino_cv,r2_teste_cv,r2_teste_separado,erro_absoluto_medio
0,"sem padronização (10, 10), alpha=1",0.842,0.834,0.764,4227.919
1,"padronizada (10,), alpha=0.01",0.872,0.839,0.856,2813.786


In [43]:
#concluimos que depois de ajustar os hiperparâmetros, as duas redes ficaram quase empatadas na validação cruzada (0,834 sem padronizar e 0,839 padronizada)
#mas no teste separado a padronizada foi bem melhor: 0,856 contra 0,764, e errou menos, 2814 de erro absoluto médio contra 4228
#então a padronização não mudou muito o melhor resultado possível, mas deixou o treino mais estável
#a melhor rede deste projeto foi a padronizada (10,) com alpha=0.01, que ficou praticamente igual ao LightGBM (0,856 contra 0,858)
#como a diferença é minúscula, eu ficaria com o LightGBM, que é mais simples e não precisa nem padronizar
#o LightGBM foi validado com 5 folds e as redes com 10, então o número mais justo para comparar os dois é o R² do teste separado